# I2 Recommendation System - Pipeline Tutorial

This notebook provides a comprehensive understanding of the I2 Recommendation System pipeline.

## Tutorial Sections
1. Setup & Introduction
2. Configuration Basics
3. Policy-Specific Configurations
4. DataManager Interface
5. Data Loading & Filtering
6. Summary & Next Steps

In [ ]:
# CELL 1: Setup - Run this first!
import sys
import os
import warnings
from pathlib import Path
import pandas as pd
import yaml
from pprint import pprint

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add i2rec_prod to path
notebook_dir = os.getcwd()
if 'i2rec_prod' not in sys.path:
    sys.path.insert(0, os.path.join(notebook_dir, 'i2rec_prod'))

# Import pipeline components with i2rec_prod prefix
from i2rec_prod.pipeline_config import PipelineConfig, PolicyInfo, DateRangeConfig
from i2rec_prod.data_manager import DataManager
from i2rec_prod.data_io import (
    DataIOConfig,
    DataLoader,
    DataValidator,
    AppealsFilter,
    CatchRateFilter,
    PlatformIssuesFilter
)
from i2rec_prod.data_io.models import FilterConfig
from i2rec_prod.data_io.config import ValidationConfig

# Suppress logging warnings
import logging
logging.getLogger('i2rec_prod').setLevel(logging.ERROR)

print("="*80)
print("✓ Setup Complete!")
print("="*80)
print(f"Working directory: {os.getcwd()}")
print("\nYou're ready to run the tutorial! Continue with the cells below.")


---
## Section 1: Introduction

The I2 Recommendation System analyzes policy violations and provides recommendations.

### System Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                     I2 Recommendation System                    │
└─────────────────────────────────────────────────────────────────┘
                                 │
                    ┌────────────┴────────────┐
                    │                         │
            ┌───────▼────────┐      ┌────────▼────────┐
            │  Configuration │      │  DataManager    │
            │     System     │◄─────┤  (Main Entry)   │
            └───────┬────────┘      └────────┬────────┘
                    │                        │
        ┌───────────┼────────────┐          │
        │           │            │          │
   ┌────▼───┐  ┌───▼────┐  ┌───▼─────┐   │
   │  Base  │  │  Env   │  │ Policy  │   │
   │ Config │  │ Config │  │ Config  │   │
   └────────┘  └────────┘  └────┬────┘   │
                                 │        │
                    ┌────────────┴────────▼────────────┐
                    │        Data Pipeline             │
                    └────────────┬─────────────────────┘
                                 │
                ┌────────────────┼────────────────┐
                │                │                │
         ┌──────▼──────┐  ┌─────▼─────┐  ┌──────▼────────┐
         │   Filters   │  │Validators │  │   Analyzers   │
         │             │  │           │  │               │
         │ - Appeals   │  │ - Schema  │  │ - LLM         │
         │ - CatchRate │  │ - Quality │  │ - RCA         │
         │ - Platform  │  │ - Business│  │ - Clustering  │
         └─────────────┘  └───────────┘  └───────────────┘
```

### Key Components:

- **Configuration System**: Manages settings for policies and environments
- **DataManager**: Central interface for all pipeline operations  
- **Filters**: Apply business logic (appeals, catch rate, platform issues)
- **Validators**: Ensure data quality
- **Analyzers**: Generate insights and recommendations

### Component Relationships

```
User/CLI Command
      │
      │ python main.py --config adult_5985.yaml
      │
      ▼
┌─────────────┐
│ main.py     │
│             │
│ - Parse CLI │
│ - Load cfg  │
└──────┬──────┘
       │
       │ DataManager.from_config()
       │
       ▼
┌─────────────────────────────┐
│      DataManager            │
│                             │
│ Responsibilities:           │
│ • Hold PipelineConfig       │
│ • Lazy load data            │
│ • Cache results             │
│ • Provide helper methods    │
└──────┬──────────────────────┘
       │
       │ Uses
       │
   ┌───┴───┐
   │       │
   ▼       ▼
┌──────┐ ┌──────────┐
│Config│ │Data I/O  │
│      │ │          │
│YAML  │ │Filters   │
│Files │ │Validators│
└──────┘ └──────────┘
```

---
## Section 2: Configuration Basics

### Configuration Hierarchy

The system uses hierarchical configs:
1. **Base Config** - Common defaults
2. **Environment Config** - Environment-specific (dev/prod)
3. **Policy Config** - Policy-specific overrides

Later configs override earlier ones using deep merge.

### Configuration Hierarchy Diagram

```
┌─────────────────────────────────────────────────────────────┐
│                    Configuration Loading                    │
└─────────────────────────────────────────────────────────────┘

Step 1: Load Base Config (Common Defaults)
┌────────────────────────────────────┐
│  config/base.yaml                  │
│                                    │
│  • LLM settings (model, temp)      │
│  • Filter defaults                 │
│  • Pipeline components             │
│  • Output directories              │
│  • Validation rules                │
└────────────────┬───────────────────┘
                 │
                 │ Deep Merge
                 ▼
Step 2: Merge Environment Config (Optional)
┌────────────────────────────────────┐
│  config/environments/dev.yaml      │
│                                    │
│  • Override LLM model (cheaper)    │
│  • Override output dir             │
│  • Override cache settings         │
└────────────────┬───────────────────┘
                 │
                 │ Deep Merge
                 ▼
Step 3: Merge Policy Config (Policy-Specific)
┌────────────────────────────────────┐
│  config/policies/adult_5985.yaml   │
│                                    │
│  • L5 Policy ID: 5985              │
│  • Date Range: Jan 2026            │
│  • Input CSV path                  │
│  • Policy-specific thresholds      │
└────────────────┬───────────────────┘
                 │
                 ▼
        ┌────────────────┐
        │ Final Config   │
        │  PipelineConfig│
        │   (Merged)     │
        └────────────────┘
```

### Configuration Class Hierarchy

```
PipelineConfig (Main Config Object)
│
├─── PolicyInfo                    # NEW! L5 Policy metadata
│    ├── l5_policy_id: int
│    ├── policy_name: str
│    ├── description: str
│    ├── policy_category: str
│    └── priority_level: str
│
├─── DateRangeConfig               # NEW! Analysis date range
│    ├── start_date: str
│    ├── end_date: str
│    └── timezone: str
│
├─── LLMConfig
│    ├── model: str
│    ├── temperature: float
│    ├── max_tokens: int
│    └── cache_enabled: bool
│
├─── DataConfig
│    ├── input_csv: str
│    ├── prompt_yaml: str
│    └── taxonomy_csv: str
│
├─── FilterConfig
│    ├── appeals_filter_granted_only: bool
│    ├── catch_rate_sample_size: int
│    └── platform_null_check_fields: list
│
├─── ValidationConfig
│    ├── enable_schema_validation: bool
│    ├── enable_quality_validation: bool
│    └── strict_validation: bool
│
├─── PipelineComponentsConfig
│    ├── enable_appeals_analysis: bool
│    ├── enable_catch_rate_analysis: bool
│    └── enable_platform_issues: bool
│
└─── OutputConfig
     ├── base_dir: str
     ├── create_timestamp_dirs: bool
     └── save_intermediate: bool
```

### How Deep Merge Works

```
Base Config:              Policy Config:           Result:
┌──────────────┐         ┌──────────────┐        ┌──────────────┐
│ llm:         │         │ llm:         │        │ llm:         │
│   model: gpt4│   +     │   temp: 0.0  │   =    │   model: gpt4│
│   temp: 0.7  │         │              │        │   temp: 0.0  │ ← Overridden
│              │         │ policy:      │        │              │
│ filters:     │         │   id: 5985   │        │ policy:      │
│   appeals: T │         │              │        │   id: 5985   │ ← Added
└──────────────┘         └──────────────┘        │              │
                                                 │ filters:     │
                                                 │   appeals: T │ ← Kept
                                                 └──────────────┘
```

In [ ]:
# Example 2.1: Load base configuration
print("="*80)
print("EXAMPLE 2.1: Loading Base Configuration")
print("="*80)

base_config = PipelineConfig.from_yaml('i2rec_prod/config/base.yaml')

print(f"\nConfig Name: {base_config.config_name}")
print(f"Environment: {base_config.environment}")
print(f"LLM Model: {base_config.llm.model}")
print(f"LLM Temperature: {base_config.llm.temperature}")
print(f"Output Directory: {base_config.output.base_dir}")

print(f"\nPipeline Components:")
print(f"  Appeals Analysis.......... {base_config.pipeline.enable_appeals_analysis}")
print(f"  Catch Rate Analysis....... {base_config.pipeline.enable_catch_rate_analysis}")
print(f"  Platform Issues........... {base_config.pipeline.enable_platform_issues}")

In [ ]:
# Example 2.2: View configuration structure
print("="*80)
print("EXAMPLE 2.2: Configuration Structure")
print("="*80)

config_dict = base_config.model_dump()

print("\nLLM Configuration:")
pprint(config_dict['llm'])

print("\nFilter Configuration:")
pprint(config_dict['filters'])

print("\nOutput Configuration:")
pprint(config_dict['output'])

---
## Section 3: Policy-Specific Configurations

Each policy config includes:
- **Policy ID & Metadata** - L5 Policy ID, name, description
- **Date Range** - Analysis period
- **Data Sources** - CSV, prompts, taxonomies
- **Policy Settings** - Thresholds, filters, validation

In [ ]:
# Example 3.1: Load Adult Content Policy (5985)
print("="*80)
print("EXAMPLE 3.1: Adult Content Policy (5985)")
print("="*80)

adult_config = PipelineConfig.from_yaml('i2rec_prod/config/policies/adult_5985.yaml')

print("\n📋 POLICY INFORMATION:")
print(f"  L5 Policy ID: {adult_config.policy.l5_policy_id}")
print(f"  Policy Name: {adult_config.policy.policy_name}")
print(f"  Description: {adult_config.policy.description}")
print(f"  Category: {adult_config.policy.policy_category}")
print(f"  Priority: {adult_config.policy.priority_level}")

print("\n📅 DATE RANGE:")
print(f"  Period: {adult_config.data_range.date_range_str}")
print(f"  Timezone: {adult_config.data_range.timezone}")

print("\n📁 DATA SOURCES:")
print(f"  Input CSV: {adult_config.data.input_csv}")
print(f"  Prompts: {adult_config.data.prompt_yaml}")

print("\n📤 OUTPUT:")
print(f"  Directory: {adult_config.output.base_dir}")

In [ ]:
# Example 3.2: Load Knives Policy (5821)
print("="*80)
print("EXAMPLE 3.2: Prohibited Knives Policy (5821)")
print("="*80)

knives_config = PipelineConfig.from_yaml('i2rec_prod/config/policies/knives_5821.yaml')

print("\n📋 POLICY INFORMATION:")
print(f"  L5 Policy ID: {knives_config.policy.l5_policy_id}")
print(f"  Policy Name: {knives_config.policy.policy_name}")
print(f"  Category: {knives_config.policy.policy_category}")
print(f"  Priority: {knives_config.policy.priority_level}")

print("\n📅 DATE RANGE:")
print(f"  Period: {knives_config.data_range.date_range_str}")

print("\n🎯 PRODUCT THRESHOLDS:")
for product_name, product_config in knives_config.product_configs.items():
    print(f"\n  {product_name}:")
    print(f"    Confidence Threshold: {product_config.confidence_threshold}")
    print(f"    Auto-Action Min: {product_config.min_confidence_for_auto_action}")
    print(f"    Subclass RCA: {product_config.enable_subclass_rca}")

In [ ]:
# Example 3.3: Compare policies
print("="*80)
print("EXAMPLE 3.3: Policy Comparison")
print("="*80)

comparison_df = pd.DataFrame({
    'Attribute': [
        'Policy ID',
        'Policy Name',
        'Category',
        'Priority',
        'Date Range',
        'Strict Validation'
    ],
    'Adult (5985)': [
        adult_config.policy.l5_policy_id,
        adult_config.policy.policy_name,
        adult_config.policy.policy_category,
        adult_config.policy.priority_level,
        adult_config.data_range.date_range_str,
        adult_config.validation.strict_validation
    ],
    'Knives (5821)': [
        knives_config.policy.l5_policy_id,
        knives_config.policy.policy_name,
        knives_config.policy.policy_category,
        knives_config.policy.priority_level,
        knives_config.data_range.date_range_str,
        knives_config.validation.strict_validation
    ]
}).set_index('Attribute')

print("\n", comparison_df)

---
## Section 4: DataManager Interface

**DataManager** is the recommended high-level interface. It provides:
- Single point of access for all data
- Lazy loading & automatic caching
- Configuration access
- Helper methods for common operations

### DataManager Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                          DataManager                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  Responsibilities:                                              │
│  • Load and hold PipelineConfig                                 │
│  • Provide lazy-loaded data access                              │
│  • Cache loaded data (avoid re-reading CSV files)               │
│  • Provide helper methods for common operations                 │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│  Key Methods:                                                   │
│                                                                 │
│  Initialization:                                                │
│    • from_config(path, env) → DataManager                       │
│    • initialize(config) → DataManager                           │
│                                                                 │
│  Configuration Access:                                          │
│    • get_policy_id() → int                                      │
│    • get_date_range() → tuple                                   │
│    • is_component_enabled(name) → bool                          │
│                                                                 │
│  Data Access (Lazy Loaded):                                     │
│    • get_raw_data() → DataFrame                                 │
│    • get_appeal_data() → DataFrame                              │
│    • get_catch_rate_data() → DataFrame                          │
│    • get_platform_issues() → DataFrame                          │
│    • get_prompt_config() → PromptConfig                         │
│    • get_taxonomy() → DataFrame                                 │
│                                                                 │
│  Validation:                                                    │
│    • validate_data() → ValidationResult                         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### DataManager Lazy Loading Pattern

```
First Call:                    Subsequent Calls:
┌──────────────┐              ┌──────────────┐
│ User calls   │              │ User calls   │
│get_raw_data()│              │get_raw_data()│
└──────┬───────┘              └──────┬───────┘
       │                             │
       ▼                             │
  ┌─────────┐                        │
  │ Cache   │                        │
  │ Empty?  │                        │
  └────┬────┘                        │
       │ Yes                         │
       ▼                             │
  ┌──────────────┐                   │
  │ Read CSV     │                   │
  │ Process data │                   │
  └──────┬───────┘                   │
         │                           │
         ▼                           ▼
    ┌─────────────┐            ┌─────────────┐
    │ Store in    │            │ Return from │
    │ _raw_data   │            │ _raw_data   │
    │ cache       │            │ cache       │
    └─────┬───────┘            └─────────────┘
          │                           ▲
          │                           │
          └───────────────────────────┘
              Return cached data
              (No re-reading!)
```

### DataManager Usage Flow

```
Step 1: Create DataManager
┌─────────────────────────────────────┐
│ manager = DataManager.from_config(  │
│     'policies/adult_5985.yaml',     │
│     env='production'                │
│ )                                   │
└────────────┬────────────────────────┘
             │
             ▼
      Loads & merges configs
             │
             ▼
┌─────────────────────────────────────┐
│  DataManager object created         │
│  • config: PipelineConfig           │
│  • _raw_data: None (not loaded yet) │
│  • _appeal_data: None               │
│  • _catch_rate_data: None           │
└────────────┬────────────────────────┘
             │
             │
Step 2: Access Data (Lazy Load)
             │
             ▼
┌─────────────────────────────────────┐
│ df = manager.get_raw_data()         │
└────────────┬────────────────────────┘
             │
             ▼
      Check if _raw_data exists
             │
         No  │
             ▼
      Read CSV file
             │
             ▼
      Store in _raw_data cache
             │
             ▼
      Return DataFrame
             │
             │
Step 3: Access Filtered Data
             │
             ▼
┌─────────────────────────────────────┐
│ df_appeals =                        │
│     manager.get_appeal_data()       │
└────────────┬────────────────────────┘
             │
             ▼
      Check if _appeal_data exists
             │
         No  │
             ▼
      Get raw data (from cache!)
             │
             ▼
      Apply AppealsFilter
             │
             ▼
      Store in _appeal_data cache
             │
             ▼
      Return filtered DataFrame
```

### When to Use DataManager vs Direct Classes

```
✓ USE DataManager (Recommended):
┌────────────────────────────────────┐
│ manager = DataManager.from_config()│
│ df = manager.get_raw_data()        │
│ df_appeals = manager.get_appeal()  │
└────────────────────────────────────┘

Benefits:
  • Single entry point
  • Automatic caching
  • Config already loaded
  • Helper methods available
  • Consistent interface

✗ USE Direct Classes (Advanced):
┌────────────────────────────────────┐
│ loader = DataLoader(config)        │
│ df = loader.load_csv(path)         │
│ filter = AppealsFilter(cfg)        │
│ df_appeals = filter.apply(df)      │
└────────────────────────────────────┘

When to use:
  • Custom pipelines
  • Unit testing
  • Performance optimization
  • Non-standard workflows
```

In [ ]:
# Example 4.1: Create DataManager
print("="*80)
print("EXAMPLE 4.1: DataManager Creation")
print("="*80)

manager = DataManager.from_config(
    'i2rec_prod/config/policies/adult_5985.yaml',
    env='development'
)

print("\n✓ DataManager created successfully!")
print(f"\nConfig: {manager.config.config_name}")
print(f"Environment: {manager.config.environment}")

In [ ]:
# Example 4.2: Access policy information
print("="*80)
print("EXAMPLE 4.2: Access Policy Information")
print("="*80)

# Using helper methods (recommended)
policy_id = manager.get_policy_id()
date_range = manager.get_date_range()

print("\n✓ Helper Methods:")
print(f"  Policy ID: {policy_id}")
print(f"  Date Range: {date_range}")

# Direct access (also available)
print("\n✓ Direct Access:")
print(f"  Policy Name: {manager.config.policy.policy_name}")
print(f"  Priority: {manager.config.policy.priority_level}")
print(f"  Date String: {manager.config.data_range.date_range_str}")

In [ ]:
# Example 4.3: Check enabled components
print("="*80)
print("EXAMPLE 4.3: Pipeline Components")
print("="*80)

components = [
    'appeals_analysis',
    'catch_rate_analysis',
    'platform_issues',
    'recommendations',
    'dashboard'
]

print("\nEnabled Components:")
for component in components:
    enabled = manager.is_component_enabled(component)
    status = "✓ Enabled" if enabled else "✗ Disabled"
    print(f"  {component:.<30} {status}")

---
## Section 5: Data Loading & Filtering

### Data Flow
1. **Raw Data** - Load from CSV
2. **Appeals Data** - Filter granted appeals (false positives)
3. **Catch Rate Data** - Filter valid items, apply sampling
4. **Platform Issues** - Identify data quality issues

### Data Pipeline Flow Diagram

```
┌────────────────────────────────────────────────────────────────┐
│                        Data Pipeline                           │
└────────────────────────────────────────────────────────────────┘

Input: CSV File
┌─────────────────────────────────┐
│  Raw Data (input CSV)           │
│                                 │
│  • ITEM_ID                      │
│  • TITLE                        │
│  • APPEAL_GRANTED_FLAG          │
│  • eval_confidence_score        │
│  • eval_matched                 │
│  • product_sub_class            │
│  • ... (other columns)          │
└────────────┬────────────────────┘
             │
             │ manager.get_raw_data()
             │
             ▼
┌─────────────────────────────────┐
│  Data Validation                │
│                                 │
│  ✓ Schema validation            │
│  ✓ Quality checks               │
│  ✓ Business rules               │
└────────────┬────────────────────┘
             │
             │ If valid, continue
             │
       ┌─────┴─────┬──────────────┬─────────────┐
       │           │              │             │
       ▼           ▼              ▼             ▼
┌──────────┐ ┌──────────┐ ┌────────────┐ ┌──────────┐
│ Appeals  │ │ Catch    │ │ Platform   │ │ Direct   │
│ Analysis │ │ Rate     │ │ Issues     │ │ Use      │
└─────┬────┘ └─────┬────┘ └──────┬─────┘ └────┬─────┘
      │            │             │            │
      ▼            ▼             ▼            ▼
┌──────────┐ ┌──────────┐ ┌────────────┐ ┌──────────┐
│Filtered  │ │Sampled   │ │Items with  │ │Full      │
│Appeals   │ │Valid     │ │Nulls/      │ │Dataset   │
│(FP)      │ │Items     │ │Issues      │ │          │
└──────────┘ └──────────┘ └────────────┘ └──────────┘
```

### Filter Class Hierarchy

```
BaseFilter (Abstract Base Class)
│
│  Methods:
│  • apply(df) → DataFrame        [ABSTRACT - must implement]
│  • validate_input(df) → None    [Check required columns]
│  • log_filter_stats(before, after) → None
│
├─── AppealsFilter
│    │
│    │  Purpose: Filter granted appeals (false positives)
│    │
│    │  Config Parameters:
│    │  • appeals_filter_granted_only: bool
│    │  • appeals_granted_value: int (default: 1)
│    │
│    │  apply() Logic:
│    │  └─► Filter rows where APPEAL_GRANTED_FLAG == 1
│    │
│    └─► Output: DataFrame with only granted appeals
│
├─── CatchRateFilter
│    │
│    │  Purpose: Sample valid items for catch rate analysis
│    │
│    │  Config Parameters:
│    │  • catch_rate_filter_valid_items: bool
│    │  • catch_rate_sample_size: int (None = all)
│    │  • catch_rate_sort_by: str (column name)
│    │
│    │  apply() Logic:
│    │  ├─► Filter valid items (eval_confidence_score_backup == 1)
│    │  ├─► Sort by specified column
│    │  └─► Sample N rows (or all if sample_size is None)
│    │
│    └─► Output: Sampled DataFrame
│
└─── PlatformIssuesFilter
     │
     │  Purpose: Identify data quality issues (nulls, missing values)
     │
     │  Config Parameters:
     │  • platform_null_check_fields: List[str]
     │
     │  apply() Logic:
     │  └─► Find rows with NULL values in specified columns
     │
     └─► Output: DataFrame with items that have issues
```

### Filter Application Flow

```
Example: Applying AppealsFilter

Step 1: Raw Data
┌────────────────────────────────────────────┐
│ ITEM_ID │ TITLE  │ APPEAL_GRANTED_FLAG     │
├─────────┼────────┼─────────────────────────┤
│ ITEM001 │ Book   │ 1  ← GRANTED            │
│ ITEM002 │ Knife  │ 1  ← GRANTED            │
│ ITEM003 │ Toy    │ 0  ← NOT GRANTED        │
│ ITEM004 │ Game   │ 1  ← GRANTED            │
│ ITEM005 │ Tool   │ 0  ← NOT GRANTED        │
└────────────────────────────────────────────┘
   5 rows

Step 2: Create Filter Config
┌────────────────────────────────────────────┐
│ filter_config = FilterConfig(              │
│     appeals_filter_granted_only=True,      │
│     appeals_granted_value=1                │
│ )                                          │
└────────────────────────────────────────────┘

Step 3: Create & Apply Filter
┌────────────────────────────────────────────┐
│ filter = AppealsFilter(filter_config)      │
│ df_appeals = filter.apply(sample_data)     │
└────────────────────────────────────────────┘

Step 4: Filtered Output
┌────────────────────────────────────────────┐
│ ITEM_ID │ TITLE  │ APPEAL_GRANTED_FLAG     │
├─────────┼────────┼─────────────────────────┤
│ ITEM001 │ Book   │ 1                       │
│ ITEM002 │ Knife  │ 1                       │
│ ITEM004 │ Game   │ 1                       │
└────────────────────────────────────────────┘
   3 rows (filtered from 5)
```

### Important Column Names Reference

```
Database Schema Columns (Use These!):
┌──────────────────────────────────────────────────────────┐
│ Column Name                  │ Type      │ Purpose       │
├──────────────────────────────┼───────────┼───────────────┤
│ ITEM_ID                      │ str       │ Item ID       │
│ TITLE (AUCT_TITL in DB)      │ str       │ Title         │
│ APPEAL_GRANTED_FLAG          │ int       │ Appeal status │
│ eval_confidence_score        │ float     │ Confidence    │
│ eval_confidence_score_backup │ int       │ Valid flag    │
│ eval_matched                 │ str       │ Matched class │
│ product_sub_class            │ str       │ Product class │
└──────────────────────────────────────────────────────────┘

Common Mistakes:
❌ GRANTED              → ✓ APPEAL_GRANTED_FLAG
❌ confidence_score     → ✓ eval_confidence_score
❌ matched_class        → ✓ eval_matched
❌ subclass             → ✓ product_sub_class
```

### Note
Since the actual data files may not exist, we'll use **sample data** for demonstration.

In [ ]:
# Example 5.1: Create sample data
print("="*80)
print("EXAMPLE 5.1: Sample Data")
print("="*80)

sample_data = pd.DataFrame({
    'ITEM_ID': ['ITEM001', 'ITEM002', 'ITEM003', 'ITEM004', 'ITEM005'],
    'TITLE': ['Adult Book', 'Kitchen Knife', 'Kids Toy', 'Tactical Knife', 'Normal Product'],
    'APPEAL_GRANTED_FLAG': [1, 1, 0, 1, 0],
    'eval_confidence_score_backup': [1, 1, 1, 0, 1],
    'eval_matched': ['Adult', 'Bladed', 'Toys', None, 'General'],
    'product_sub_class': ['Books', 'Kitchen', 'Games', None, 'Home'],
    'eval_confidence_score': [0.95, 0.88, 0.92, 0.75, 0.98]
})

print("\nSample Data:")
print(sample_data)

print(f"\n✓ Stats:")
print(f"  Total rows: {len(sample_data)}")
print(f"  Granted appeals: {sample_data['APPEAL_GRANTED_FLAG'].sum()}")
print(f"  Items with confidence > 0.8: {4}")
print(f"  Null values in eval_matched: {sample_data['eval_matched'].isna().sum()}")

In [ ]:
# Example 5.2: Apply Appeals Filter
print("="*80)
print("EXAMPLE 5.2: Appeals Filtering")
print("="*80)

# Create filter
filter_config = FilterConfig(
    appeals_filter_granted_only=True,
    appeals_granted_value=1
)
appeals_filter = AppealsFilter(filter_config)

# Apply filter
df_appeals = appeals_filter.apply(sample_data)

print(f"\nOriginal rows: {len(sample_data)}")
print(f"After filter: {len(df_appeals)}")
print(f"\n✓ Filtered Data (Granted Appeals Only):")
print(df_appeals[['ITEM_ID', 'TITLE', 'APPEAL_GRANTED_FLAG', 'eval_matched']])

In [ ]:
# Example 5.3: Apply Catch Rate Filter
print("="*80)
print("EXAMPLE 5.3: Catch Rate Filtering")
print("="*80)

# Create filter
filter_config = FilterConfig(
    catch_rate_filter_valid_items=True,
    catch_rate_sample_size=None,
    catch_rate_sort_by='eval_confidence_score'
)
catch_rate_filter = CatchRateFilter(filter_config)

# Apply filter
df_catch_rate = catch_rate_filter.apply(sample_data)

print(f"\nOriginal rows: {len(sample_data)}")
print(f"After filter: {len(df_catch_rate)}")
print(f"\n✓ Filtered Data (Valid Items, Sorted by Confidence):")
print(df_catch_rate[['ITEM_ID', 'TITLE', 'eval_confidence_score', 'eval_confidence_score']])

In [ ]:
# Example 5.4: Identify Platform Issues
print("="*80)
print("EXAMPLE 5.4: Platform Issues Detection")
print("="*80)

# Create filter
filter_config = FilterConfig(
    platform_null_check_fields=['ITEM_ID', 'eval_matched', 'product_sub_class']
)
platform_filter = PlatformIssuesFilter(filter_config)

# Apply filter
df_issues = platform_filter.apply(sample_data)

print(f"\nOriginal rows: {len(sample_data)}")
print(f"Issues found: {len(df_issues)}")
print(f"\n✓ Items with Null Values:")
print(df_issues[['ITEM_ID', 'TITLE', 'eval_matched', 'product_sub_class']])

---
## Section 6: Summary

### What You Learned

✓ **Configuration Hierarchy** - Base → Environment → Policy configs

✓ **Policy Configs** - L5 Policy ID, date ranges, policy-specific settings

✓ **DataManager** - High-level interface for all pipeline operations

✓ **Filters** - Appeals, catch rate, and platform issues filters

### Complete End-to-End Workflow Diagram

```
┌───────────────────────────────────────────────────────────────────────┐
│                    COMPLETE WORKFLOW - START TO FINISH                │
└───────────────────────────────────────────────────────────────────────┘

STEP 1: USER INITIATES ANALYSIS
┌─────────────────────────────────────────┐
│ Terminal:                               │
│ $ python main.py --config adult_5985.yaml│
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 2: MAIN.PY LOADS CONFIGURATION
┌─────────────────────────────────────────┐
│ main.py:                                │
│                                         │
│ 1. Parse CLI args                       │
│ 2. Resolve config path                  │
│ 3. Call DataManager.from_config()       │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 3: CONFIG HIERARCHY MERGE
┌─────────────────────────────────────────┐
│ PipelineConfig.from_yaml():             │
│                                         │
│ Load: base.yaml                         │
│   ↓ Merge                               │
│ Load: environments/production.yaml      │
│   ↓ Merge                               │
│ Load: policies/adult_5985.yaml          │
│   ↓                                     │
│ Result: Final merged config             │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 4: DATAMANAGER INITIALIZED
┌─────────────────────────────────────────┐
│ DataManager object created:             │
│                                         │
│ • config: PipelineConfig                │
│ • _raw_data: None (lazy)                │
│ • _appeal_data: None (lazy)             │
│ • _catch_rate_data: None (lazy)         │
│                                         │
│ Display policy info:                    │
│ • L5 Policy ID: 5985                    │
│ • Policy Name: Adult Content            │
│ • Date Range: 2026-01-01 to 2026-01-31  │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 5: LOAD & VALIDATE DATA
┌─────────────────────────────────────────┐
│ df = manager.get_raw_data()             │
│   ↓                                     │
│ DataLoader reads CSV                    │
│   ↓                                     │
│ Store in _raw_data cache                │
│   ↓                                     │
│ validator.validate_data(df)             │
│   ├─ Schema validation                  │
│   ├─ Quality validation                 │
│   └─ Business validation                │
│   ↓                                     │
│ Validation passed ✓                     │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 6: APPLY FILTERS (IF ENABLED)
┌─────────────────────────────────────────┐
│ If config.pipeline.enable_appeals:      │
│   df_appeals = manager.get_appeal_data()│
│     ↓                                   │
│   AppealsFilter.apply(raw_data)         │
│     ↓                                   │
│   Cache result in _appeal_data          │
│                                         │
│ If config.pipeline.enable_catch_rate:   │
│   df_catch = manager.get_catch_rate()   │
│     ↓                                   │
│   CatchRateFilter.apply(raw_data)       │
│     ↓                                   │
│   Cache result in _catch_rate_data      │
│                                         │
│ If config.pipeline.enable_platform:     │
│   df_issues = manager.get_platform()    │
│     ↓                                   │
│   PlatformIssuesFilter.apply(raw_data)  │
│     ↓                                   │
│   Cache result                          │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 7: ANALYSIS & RECOMMENDATIONS
┌─────────────────────────────────────────┐
│ For each enabled analyzer:              │
│                                         │
│ 1. LLM Analysis (if enabled)            │
│    • Load prompts from YAML             │
│    • Call LLM API                       │
│    • Parse recommendations              │
│                                         │
│ 2. RCA (Root Cause Analysis)            │
│    • Identify patterns                  │
│    • Group by product_sub_class         │
│    • Generate insights                  │
│                                         │
│ 3. Clustering (if enabled)              │
│    • Cluster similar items              │
│    • Identify anomalies                 │
└──────────────┬──────────────────────────┘
               │
               ▼
STEP 8: OUTPUT RESULTS
┌─────────────────────────────────────────┐
│ Save to: config.output.base_dir         │
│                                         │
│ Files created:                          │
│ • appeals_analysis.csv                  │
│ • catch_rate_analysis.csv               │
│ • platform_issues.csv                   │
│ • recommendations.json                  │
│ • dashboard.html (if enabled)           │
│                                         │
│ Log summary:                            │
│ ✓ Policy 5985 analysis complete         │
│ ✓ Date range: 2026-01-01 to 2026-01-31  │
│ ✓ Output: output/adult_5985/            │
└─────────────────────────────────────────┘
```

### Key Design Patterns Used

```
┌──────────────────────────────────────────────────────────────┐
│ 1. FACADE PATTERN                                            │
├──────────────────────────────────────────────────────────────┤
│ DataManager acts as a simplified interface to complex       │
│ subsystems (config, loaders, filters, validators)           │
│                                                              │
│ Instead of:                    Use:                          │
│ config = load_config()        manager = DataManager()       │
│ loader = DataLoader(config)   df = manager.get_raw_data()   │
│ df = loader.load()                                           │
└──────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────┐
│ 2. LAZY LOADING PATTERN                                      │
├──────────────────────────────────────────────────────────────┤
│ Data is only loaded when first accessed, then cached        │
│                                                              │
│ Benefits:                                                    │
│ • Faster initialization                                      │
│ • Lower memory usage                                         │
│ • Only load what you need                                    │
└──────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────┐
│ 3. STRATEGY PATTERN                                          │
├──────────────────────────────────────────────────────────────┤
│ Different filter strategies (Appeals, CatchRate, Platform)   │
│ all implement common interface (apply method)               │
│                                                              │
│ Benefits:                                                    │
│ • Easy to add new filters                                    │
│ • Consistent interface                                       │
│ • Testable in isolation                                      │
└──────────────────────────────────────────────────────────────┘

┌──────────────────────────────────────────────────────────────┐
│ 4. CONFIGURATION HIERARCHY (Cascading Configs)               │
├──────────────────────────────────────────────────────────────┤
│ Base → Environment → Policy configs merged using deep merge │
│                                                              │
│ Benefits:                                                    │
│ • DRY - no duplication                                       │
│ • Easy to override                                           │
│ • Clear precedence rules                                     │
└──────────────────────────────────────────────────────────────┘
```

### Quick Reference

```python
# Load policy config
manager = DataManager.from_config('i2rec_prod/config/policies/adult_5985.yaml')

# Access policy info
policy_id = manager.get_policy_id()  # 5985
date_range = manager.get_date_range()  # ('2026-01-01', '2026-01-31')

# Check components
if manager.is_component_enabled('appeals_analysis'):
    # Run appeals analysis
    pass

# Get data (lazy loaded & cached)
df_raw = manager.get_raw_data()
df_appeals = manager.get_appeal_data()
df_catch_rate = manager.get_catch_rate_data()
df_issues = manager.get_platform_issues()

# Validate data
result = manager.validate_data()
if result.has_errors:
    print(f"Errors: {result.errors}")
```

### Common Tasks for Junior Engineers

```
Task 1: Add a New Policy Config
┌──────────────────────────────────────────┐
│ 1. Copy existing policy YAML            │
│ 2. Update policy ID and name             │
│ 3. Update date range                     │
│ 4. Update data paths (CSV, prompts)      │
│ 5. Adjust thresholds if needed           │
│ 6. Test: python main.py --config <new>  │
└──────────────────────────────────────────┘

Task 2: Add a New Filter
┌──────────────────────────────────────────┐
│ 1. Create class inheriting BaseFilter   │
│ 2. Implement apply(df) method            │
│ 3. Add config parameters to FilterConfig │
│ 4. Add filter to DataManager             │
│ 5. Write unit tests                      │
│ 6. Update documentation                  │
└──────────────────────────────────────────┘

Task 3: Modify Configuration Hierarchy
┌──────────────────────────────────────────┐
│ 1. Identify which level to modify:      │
│    • Base: affects ALL policies          │
│    • Env: affects one environment        │
│    • Policy: affects one policy only     │
│ 2. Edit appropriate YAML file            │
│ 3. Test with: python main.py --config   │
│ 4. Verify no regressions                 │
└──────────────────────────────────────────┘
```

### Next Steps

1. **Create your policy config** - Copy an existing policy YAML and customize it
2. **Run with real data** - Add your actual CSV files
3. **Customize filters** - Adjust thresholds and sampling
4. **Explore validation** - Use DataValidator for data quality
5. **Build dashboards** - Visualize your results

### Documentation

- `i2rec_prod/config/policies/README.md` - Policy config guide
- `POLICY_CONFIG_IMPLEMENTATION.md` - Implementation details
- `QUICK_START_POLICY_CONFIGS.md` - Quick reference
- `i2rec_prod/data_io/README.md` - Data I/O documentation

### Pro Tips for Junior Engineers

```
✓ Always use DataManager (not direct classes)
✓ Use helper methods (get_policy_id, get_date_range)
✓ Check is_component_enabled() before running analysis
✓ Use correct column names (APPEAL_GRANTED_FLAG not GRANTED)
✓ Filters use .apply() method (not .filter())
✓ Data is cached - calling get_raw_data() twice doesn't reload CSV
✓ Policy configs inherit from base.yaml (DRY principle)
✓ Test with sample data first before using production data
```

In [ ]:
# Tutorial Complete!
print("="*80)
print("🎉 TUTORIAL COMPLETE!")
print("="*80)

print("""
You've successfully completed the I2 Recommendation System tutorial!

✓ Configuration system
✓ Policy-specific configs
✓ DataManager interface
✓ Data filtering

Next: Create your own policy config and run with real data!

Happy analyzing! 🚀
""")